# Pilot 4

- Upbringing (Good/Bad) × Environment (Community/Family)
- Blame/Praise (–6 to +6)
- N = 276

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Standard library imports
import warnings

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from patsy.contrasts import Sum
from statsmodels.formula.api import ols


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########

# Gender
labels_gender = {
    1: 'Female',
    2: 'Male',
    3: 'Non-binary',
    4: 'Prefer not to say'
}


#################
# VISUALIZATION #
#################

# Colors
palette_main = {
    "Community": "#D55E00",
    "Family":    "#0072B2"
}

## Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_4_data.csv')

# Define factors
factors_map = {
    'racism.12-pAction': ('Bad',  'Community'),
    '12.2':              ('Bad',  'Family'),
    'racism.24-pAction': ('Good', 'Community'),
    '24.2':              ('Good', 'Family')
}

# Reshape wide to long
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for col, (upbringing, environment) in factors_map.items():
        if col in df.columns and pd.notna(row[col]):
            val = pd.to_numeric(row[col], errors = 'coerce')
            val = val - 7
            
            long_data.append({
                'ID':          p_id,
                'Upbringing':  upbringing,
                'Environment': environment,
                'Blame':       val,
                'Gender':      row.get('Gender'),
                'Age':         row.get('Age'),
                'Education':   row.get('Education')
            })

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} observations).")

## Calculate Sociodemographics

In [ ]:
# Prepare variables
df['Gender_Labeled'] = df['Gender'].map(labels_gender)
df['Age'] = pd.to_numeric(df['Age'], errors = 'coerce')

# Calculate sociodemographics and display results
for col in ['Gender_Labeled']:
    print(f"\n{col}:")
    display(df[col].value_counts().to_frame('Frequency'))

print("\nAge:")
display(df['Age'].describe().to_frame('Value').round(3))

print("\nParticipants:")
total_participants = df_long['ID'].nunique()
display(f"N = {total_participants}")

## Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variable
group_factors = ['Upbringing', 'Environment']
dependent_var = 'Blame'

# Calculate descriptive statistics
descriptive_stats = (
    df_long
    .groupby(group_factors)[dependent_var]
    .agg(['mean', 'std', 'count'])
    .round(3)
)

# Display results
display(descriptive_stats)

## Perform ANOVA

In [ ]:
dv = 'Blame'
print(f"ANOVA ({dv})")

# Define formula
formula = f"{dv} ~ C(Upbringing, Sum) * C(Environment, Sum)"

# Drop lines with NaN
df_anova = df_long.dropna(subset = [dv])

# Fit model
model = ols(formula, data = df_anova).fit()

# Run ANOVA
aov_table = sm.stats.anova_lm(model, typ = 3)

# Calculate effect sizes
aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])

# Display results
display(aov_table.round(3))

## Generate Histograms

In [ ]:
# Set condition labels
df_long['Condition'] = (df_long['Upbringing'] + " / " +
                        df_long['Environment']
                       )

# Generate graph
g = sns.displot(data     = df_long,
                x        = dependent_var,
                col      = "Condition",
                col_wrap = 4,
                kind     = "hist",
                discrete = True,
                shrink   = 0.8,
                height   = 3,
                aspect   = 1.2,
                color    = "#0072B2"
               )

# Set axis labels and subplot titles
g.set_axis_labels("Value", "Count")
g.set_titles("{col_name}")

# Limit x-axis
for ax in g.axes.flat:
    ax.set_xticks(range(-6, 7))
    ax.set_xlim(-6.5, 6.5)

# Add main title
plt.subplots_adjust(top = 0.8)
title_text = dependent_var
g.fig.suptitle(f'{title_text}', fontsize = 16)

# Export graph
filename = 'blame_praise_self_pilot_4_histograms.png'
g.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")

## Generate Bar Plot

In [ ]:
# Generate graph
g = sns.catplot(data    = df_long,
                x       = "Upbringing",
                y       = "Blame",
                hue     = "Environment",
                kind    = "bar",
                palette = palette_main,
                capsize = 0.05,
                height  = 5,
               )

# Set axis labels
plt.ylabel('Mean')
plt.xlabel('Upbringing')

# Add main title
plt.title('Blame or Praise', fontsize = 16)

# Add horizontal zero lines and limit y-axis
for ax in g.axes.flat:
    ax.axhline(0, color = 'black', lw = 1)
    ax.set_ylim(-6, 6)

# Export graph
filename = 'blame_analysis_pilot_4_bar_plot.png'
plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")